
### How agents remember conversations across multiple turns

**What you'll learn:**
- Why memory is needed
- How `MemorySaver` works
- What `thread_id` does
- Memory isolation between sessions
- Inspecting stored memory

---
> **Prerequisite:** Complete Notebook 1 — Tools first!
---

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


True

---
## 🤔 Why Memory?

By default, every LLM call is **stateless** — it forgets everything.

**Without Memory:**
```
Turn 1 — User: "My name is Rahul"
          Bot:  "Nice to meet you, Rahul!"

Turn 2 — User: "What is my name?"
          Bot:  "I don't know your name" ❌
```

**With Memory (MemorySaver):**
```
Turn 1 — User: "My name is Rahul"
          Bot:  "Nice to meet you, Rahul!"  → saved to memory

Turn 2 — User: "What is my name?"
          Bot:  "Your name is Rahul" ✅
```

Memory = **full message history injected into every LLM call.**

---
## 🏗️ Build a Simple Agent WITH and WITHOUT Memory

In [2]:
from typing import Annotated, Sequence
import operator
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

def agent_node(state: AgentState):
    response = llm.invoke(list(state["messages"]))
    return {"messages": [response]}

# ─────────────────────────────────
# Agent WITHOUT Memory
# ─────────────────────────────────
def build_agent_no_memory():
    builder = StateGraph(AgentState)
    builder.add_node("agent", agent_node)
    builder.add_edge(START, "agent")
    builder.add_edge("agent", END)
    return builder.compile()          # no checkpointer!

# ─────────────────────────────────
# Agent WITH Memory
# ─────────────────────────────────
def build_agent_with_memory():
    builder = StateGraph(AgentState)
    builder.add_node("agent", agent_node)
    builder.add_edge(START, "agent")
    builder.add_edge("agent", END)
    return builder.compile(checkpointer=MemorySaver())  # ← memory added!

app_no_mem   = build_agent_no_memory()
app_with_mem = build_agent_with_memory()
print("✅ Both agents compiled!")

✅ Both agents compiled!


---
## ⚔️ Side-by-Side Comparison

In [3]:
config = {"configurable": {"thread_id": "test_memory"}}

# ── Turn 1: Introduce yourself ──
msg1 = HumanMessage(content="My name is Rahul and I live in Mumbai.")

app_no_mem.invoke({"messages": [msg1]})
app_with_mem.invoke({"messages": [msg1]}, config=config)

# ── Turn 2: Ask about name ──
msg2 = HumanMessage(content="What is my name and where do I live?")

r_no_mem   = app_no_mem.invoke({"messages": [msg2]})
r_with_mem = app_with_mem.invoke({"messages": [msg2]}, config=config)

print("❌ WITHOUT Memory:")
print(r_no_mem["messages"][-1].content)

print()
print("✅ WITH Memory:")
print(r_with_mem["messages"][-1].content)

❌ WITHOUT Memory:
I'm sorry, but I do not have access to personal information about users.

✅ WITH Memory:
Your name is Rahul and you live in Mumbai.


---
## 🔑 What is `thread_id`?

`thread_id` is like a **conversation ID** — each unique ID gets its own independent memory.

```
thread_id = "user_rahul"    → Rahul's conversation history
thread_id = "user_priya"    → Priya's conversation history  
thread_id = "support_123"   → Support ticket #123 history
```

Same `thread_id` = **continues same conversation**  
New `thread_id` = **fresh conversation, no memory**

In [4]:
def chat(app, user_input: str, thread_id: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app_with_mem.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config
    )
    return result["messages"][-1].content

# ── thread_id = "rahul" ──────────────────────────
chat(app_with_mem, "My name is Rahul.",          thread_id="rahul")
chat(app_with_mem, "I work at EY as an ML engineer.", thread_id="rahul")
ans_rahul = chat(app_with_mem, "What do you know about me?", thread_id="rahul")

# ── thread_id = "priya" — completely separate ────
chat(app_with_mem, "My name is Priya.",          thread_id="priya")
ans_priya = chat(app_with_mem, "What do you know about me?", thread_id="priya")

print("🔵 Rahul's session:")
print(ans_rahul)
print()
print("🟢 Priya's session:")
print(ans_priya)

🔵 Rahul's session:
As an AI assistant, I don't have access to personal information about individuals unless it has been shared with me during our conversation. I am designed to respect user privacy and confidentiality. Is there anything specific you would like to share or discuss with me?

🟢 Priya's session:
I'm sorry, but I don't have access to personal information about individuals unless it has been shared with me during our conversation. How can I help you today?


---
## 🔍 Memory Isolation Test

Can Rahul's session access Priya's info? **No!** They are completely isolated.

In [ ]:
# Rahul's session asks about Priya
ans = chat(app_with_mem, "Do you know someone named Priya?", thread_id="rahul")
print("Rahul's session asking about Priya:")
print(ans)   # Should say No / doesn't know ✅

---
## 🔬 How Memory Works Internally

`MemorySaver` stores a **checkpointed state** after every node execution.

When you call `invoke()` with the same `thread_id`, it:
1. Loads the last checkpoint for that `thread_id`
2. Appends your new message
3. Runs the graph
4. Saves the updated state as a new checkpoint

Let's inspect the stored state:

In [5]:
config = {"configurable": {"thread_id": "rahul"}}

# Get the current stored state
state = app_with_mem.get_state(config)

print(f"Total messages stored: {len(state.values['messages'])}")
print()
for i, msg in enumerate(state.values["messages"]):
    role = type(msg).__name__
    print(f"[{i}] {role}: {msg.content[:80]}")

Total messages stored: 6

[0] HumanMessage: My name is Rahul.
[1] AIMessage: Hello Rahul! How can I assist you today?
[2] HumanMessage: I work at EY as an ML engineer.
[3] AIMessage: That's great to hear! Working as a Machine Learning engineer at EY must be an ex
[4] HumanMessage: What do you know about me?
[5] AIMessage: As an AI assistant, I don't have access to personal information about individual


---
## 📊 Multi-Turn Memory Demo

In [6]:
SESSION = "memory_test_session"

turns = [
    "My name is Arjun and I am from Pune.",
    "I am planning a trip to Goa next month.",
    "My budget is around 15000 INR.",
    "What do you know about my travel plans?",   # memory test
    "What is my name and where am I from?",      # memory test
    "Based on my budget, how many nights can I afford if hotel costs 2500/night?",
]

for i, turn in enumerate(turns, 1):
    config = {"configurable": {"thread_id": SESSION}}
    result = app_with_mem.invoke(
        {"messages": [HumanMessage(content=turn)]},
        config=config
    )
    print(f"\n{'─'*55}")
    print(f"[Turn {i}] 👤 {turn}")
    print(f"         🤖 {result['messages'][-1].content}")


───────────────────────────────────────────────────────
[Turn 1] 👤 My name is Arjun and I am from Pune.
         🤖 Nice to meet you, Arjun! How can I assist you today?

───────────────────────────────────────────────────────
[Turn 2] 👤 I am planning a trip to Goa next month.
         🤖 That sounds like a great trip! Goa is a beautiful destination with stunning beaches and vibrant culture. Do you need any recommendations or tips for your trip?

───────────────────────────────────────────────────────
[Turn 3] 👤 My budget is around 15000 INR.
         🤖 That's a reasonable budget for a trip to Goa. Here are some tips to help you stay within your budget:

1. Accommodation: Look for budget-friendly guesthouses, hostels, or homestays instead of expensive hotels. You can also consider booking through online platforms for discounts.

2. Transportation: Use local buses, rent a scooter, or share a cab with other travelers to save on transportation costs. Avoid taking taxis as they can be expens

---
## 🏋️ Exercises

1. Create two sessions `session_A` and `session_B` with different user names — verify they don't share memory
2. Use `app.get_state(config)` to inspect stored messages after 3 turns
3. What happens if you DON'T pass `config` to an agent with `MemorySaver`? Try it!
4. How many messages are stored after 5 turns? Count them using `get_state()`

---
**Next → Notebook 3: Full Agent** 🤖